# FishMol — Module Reference

A self-contained reference notebook for every analysis module in **FishMol**.
All classes and functions are imported directly from the installed package.

| Section | Module | Class / Function |
|---------|--------|------------------|
| 1 | Trajectory I/O | `trj.Trajectory`, `sel_tools.cluster` |
| 2 | Radial Distribution Function | `funcs.RDF` |
| 3 | Angular Distribution Function | `funcs.ADF` |
| 4 | Dihedral Distribution Function | `funcs.DDF` |
| 5 | Combined Distribution Function | `funcs.CDF` |
| 6 | Scalar Distribution Function | `funcs.SDF` |
| 7 | Vector Reorientation Dynamics | `funcs.VRD` |
| 8 | Van Hove Correlation Function | `funcs.VHCF` |
| 9 | Dimer Analysis | `dimer.dimers`, `dimer.DLDDF` |
| 10 | H-bond Lifetime | `dimer.auto_cor`, `dimer.bi_exp_fit`, `dimer.calc_tau` |


## Verify installation

Run the entry-point to confirm FishMol is installed correctly.

In [2]:
!fishmol

/mnt/d/repositories/fishmol/fishmol/__main__.py:6: SyntaxWarning: invalid escape sequence '\ '
  ascii_fishmol = """\
                                                                                       
                  Welcome!                    ▄▄█▀                  FishMol
                                          ▄▄███▀                 version 0.1.0
      ○                                ▄▄█████▀                          ○
           ○                        ▄▄████████▄                         /
                                ▄▄▄████████████▄                    ○--○
         ○                ▄▄▄████████████████████▄▄                     \ 
                    ▄▄▄██████▀███████████████████████▄▄                  ○--○           ▄
                ▄▄████████████ ██████████████████████████▄▄             /           ▄▄█▀
         ○   ▄█████████████████ ████████████████████████████▄▄         ○         ▄███▀
          ▄██████████▀▀▀████████ ██████████████████████████████▄▄           

## Imports

In [1]:
from fishmol import trj, funcs, dimer
from fishmol import vis                # optional -- requires ASE
from fishmol.sel_tools import cluster
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

plt.rcParams.update({
    "font.size": 11,
    "font.family": "sans-serif",
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.figsize": (4.2, 3.6),
    "figure.subplot.left": 0.21,
    "figure.subplot.right": 0.96,
    "figure.subplot.bottom": 0.18,
    "figure.subplot.top": 0.93,
    "legend.frameon": False,
})


---
## 1. Trajectory I/O

### Theory

FishMol reads **extended XYZ** files — the standard plain-text trajectory format produced by AIMD codes (CP2K, FHI-aims) and classical MD engines (LAMMPS, GROMACS via conversion tools).

The simulation cell is described by three lattice vectors $\mathbf{a}_1, \mathbf{a}_2, \mathbf{a}_3$ (passed as rows of a $3\times 3$ matrix, in Angstrom).  For a triclinic cell:

$$\mathbf{h} = \begin{pmatrix} a_{1x} & a_{1y} & a_{1z} \\
a_{2x} & a_{2y} & a_{2z} \\
a_{3x} & a_{3y} & a_{3z} \end{pmatrix}$$

Fractional coordinates $\mathbf{s}$ and Cartesian positions $\mathbf{r}$ are related by

$$\mathbf{r} = \mathbf{s}\,\mathbf{h}, \qquad \mathbf{s} = \mathbf{r}\,\mathbf{h}^{-1}$$

The **minimum image convention (MIC)** maps the fractional-coordinate separation to $[-0.5, 0.5)$ before converting back to Cartesian, so that all pairwise distances are measured to the nearest periodic image.  File reading uses `mmap` for fast memory-mapped I/O.

### Load a trajectory

### `.lammpstrj`

In [2]:
traj=trj.Trajectory(data='/mnt/d/repositories/fishmol/test_lammps/Ti0.1Zr0.9Mn0.9V0.1Fe0.5Ni0.5_npt.lammpstrj')

In [3]:
traj.frames[0].timestamp

1000.0

In [4]:
traj.frames[0].positions

array([[ 1.02729e+00,  3.49260e+00,  6.02879e+00],
       [ 1.39401e+00,  7.70697e-01,  2.04333e+00],
       [ 3.97474e+00,  5.99413e-01,  1.73790e+00],
       [ 1.80226e-02,  1.23863e+00,  5.95910e+00],
       [-1.15553e+00,  3.49112e+00,  6.01092e+00],
       [ 2.60343e+00,  2.82503e+00,  2.08713e+00],
       [-4.84943e+00,  8.51610e+00,  3.63609e+00],
       [ 2.28969e-02,  4.01139e-03,  1.53907e-01],
       [-1.27934e+00,  7.78661e+00,  5.95028e+00],
       [-9.68004e-01,  4.93080e+00,  2.03762e+00],
       [ 1.58500e+00,  5.00976e+00,  2.23646e+00],
       [-2.71990e+00,  5.67647e+00,  5.70798e+00],
       [-3.84559e+00,  7.73594e+00,  5.76804e+00],
       [ 2.97138e-01,  7.10294e+00,  2.16870e+00],
       [ 7.44912e+00,  4.16952e+00,  4.00569e+00],
       [-2.29237e+00,  4.22841e+00,  7.72331e+00],
       [ 6.00495e+00,  3.58585e+00,  5.71916e+00],
       [ 6.28948e+00,  8.62453e-01,  2.07737e+00],
       [ 8.73627e+00,  7.26461e-01,  2.03563e+00],
       [ 5.03338e+00,  1.51182e

In [5]:
traj.frames[0].symbs

array(['Fe', 'Fe', 'Fe', 'V', 'Mn', 'Ni', 'Mn', 'Mn', 'Ni', 'Ni', 'Mn',
       'Mn', 'Fe', 'Ni', 'Mn', 'Fe', 'V', 'Mn', 'Mn', 'Mn', 'Mn', 'Mn',
       'Ni', 'Mn', 'Ni', 'Fe', 'Ni', 'Fe', 'Mn', 'Mn', 'Fe', 'Ni', 'Zr',
       'Zr', 'Zr', 'Ti', 'Zr', 'Zr', 'Zr', 'Zr', 'Zr', 'Zr', 'Zr', 'Zr',
       'Zr', 'Zr', 'Ti', 'Zr'], dtype='<U2')

### `.xyz`

In [ ]:
# Lattice vectors as rows (in Angstrom) -- edit to match your system
cell = [
    [21.2944,  0.0000,  0.0000],
    [-4.6030, 20.7909,  0.0000],
    [-0.9719, -1.2106, 15.1054],
]

# Load all frames (index=':'), timestep 5 fs
traj = trj.Trajectory(timestep=5, data='trajectory.xyz', index=':', cell=cell)
print(f'Frames : {traj.nframes}')
print(f'Atoms  : {traj.natoms}')


### Index and slice

`Trajectory` supports Python-style indexing.  Slicing by step $k$ automatically adjusts the effective timestep to `timestep * k`.

In [ ]:
# Single frame -> 1-frame Trajectory
frame_0 = traj[0]

# Slice: every 10th frame between frame 100 and 500
sub = traj[100:500:10]
print(f'Sub-trajectory: {sub.nframes} frames, effective timestep {sub.timestep} fs')

# Arbitrary list of frame indices
pick = traj[[0, 50, 100, 150, 200]]
print(f'Custom selection: {pick.nframes} frames')


### Centre-of-mass drift correction and periodic wrapping

`calib()` removes net CoM drift by shifting every frame so its CoM matches frame 0.  This is required before MSD or diffusion calculations.

`wrap2box()` folds all Cartesian positions into the primary unit cell (convert to fractional, apply mod 1, convert back).

In [ ]:
traj.calib()      # remove CoM drift in-place; returns self for chaining
traj.wrap2box()   # wrap into [0, 1) fractional coordinates


### Visualise a frame (requires ASE)

In [ ]:
# Interactive 3-D viewer embedded in Jupyter
vis.render_atoms(traj.frames[0])


### Write a trajectory

Writes the trajectory (or any slice) back to extended XYZ.  FishMol appends a `-1` suffix if the target file already exists.

In [ ]:
traj.write(filename='output.xyz')

# Write every 5th frame of the first 1000 to a separate file
traj[:1000:5].write(filename='output_thin.xyz')


### Build atom-index groups from element symbols

Most analysis functions accept lists of atom indices.  Derive them from `frame.symbs` rather than hardcoding:

In [ ]:
symbs = traj.frames[0].symbs   # numpy array of element symbols

o_idx = [i for i, s in enumerate(symbs) if s == 'O']
h_idx = [i for i, s in enumerate(symbs) if s == 'H']
n_idx = [i for i, s in enumerate(symbs) if s == 'N']
c_idx = [i for i, s in enumerate(symbs) if s == 'C']

print(f'O: {len(o_idx)}  H: {len(h_idx)}  N: {len(n_idx)}  C: {len(c_idx)}')


### Per-frame geometry: distances, angles, dihedrals

Each `traj.frames[i]` is an `Atoms` object with `dist`, `angle`, `dihedral`, and `vec` methods, all accepting a `mic=True` flag for periodic systems.

In [ ]:
frame = traj.frames[0]

# D(A, B) under MIC
r01 = frame.dist(o_idx[0], o_idx[1], mic=True)
print(f'r(O0, O1)             = {r01:.4f} A')

# Bond angle: atoms i-j-k, vertex at j
theta = frame.angle(o_idx[0], h_idx[0], o_idx[1], mic=True)
print(f'angle(O0-H0-O1)       = {theta:.2f} deg')

# Dihedral: atoms i-j-k-l
phi = frame.dihedral(c_idx[0], c_idx[1], c_idx[2], c_idx[3], mic=True)
print(f'dihedral(C0-C1-C2-C3) = {phi:.2f} deg')


### Molecule recognition

`cluster()` builds a bond graph using covalent-radius thresholds ($r_{ij} < 1.05(R_i + R_j)$) and finds connected components by iterative DFS.  Each returned `Molecule` has `.formula` (sorted Hill notation) and `.at_idx`.

In [ ]:
molecules = cluster(traj.frames[0])
print(f'Found {len(molecules)} molecules')
print(f'First molecule: {molecules[0].formula}, indices: {molecules[0].at_idx}')

from collections import Counter
for formula, count in Counter(m.formula for m in molecules).items():
    print(f'  {formula}: {count}')


---
## 2. Radial Distribution Function (RDF)

### Theory

The RDF $g_{AB}(r)$ gives the probability of finding an atom of type $B$ at distance $r$ from a reference atom of type $A$, normalised by the uniform bulk density:

$$g_{AB}(r) = \frac{\mathrm{count}(r,r+\Delta r)}
{N_A\,N_B\,\dfrac{4\pi r^2\Delta r}{V}\,N_{\mathrm{frames}}}$$

where $V = \det(\mathbf{h})$ is the box volume.  The normalisation ensures $g_{AB}(r)\to 1$ for large $r$ in a homogeneous bulk.

**Key quantities**

| Observable | How to extract |
|------------|----------------|
| Nearest-neighbour distance | position of first peak |
| Average coordination number | $\bar{n} = 4\pi\rho_B\int_0^{r_{\min}}r^2 g(r)\,dr$ |
| Structural correlation length | $r$ where $g(r)$ decays to 1 |

In [ ]:
rdf = funcs.RDF(
    traj,
    at_g1=o_idx,    # reference group
    at_g2=h_idx,    # neighbour group
    nbins=200,
    range=(0.0, 8.0),
)
rdf_results = rdf.calculate(plot=True)


In [ ]:
# Combined panel: time-series of distances (left) + g(r) (right)
rdf_results = rdf.calculate(com_plot=True)

r_peak = rdf_results.x[rdf_results.rdf.argmax()]
print(f'First-peak position : {r_peak:.3f} A')
print(f'First-peak height   : {rdf_results.rdf.max():.3f}')


---
## 3. Angular Distribution Function (ADF)

### Theory

The ADF $g(\theta)$ is the histogram of three-body angles $\theta_{A_1\!-\!A_2\!-\!A_3}$ (vertex at $A_2$), normalised by the solid-angle element:

$$g(\theta) = \frac{\mathrm{count}(\theta,\theta+\Delta\theta)}{N_{\mathrm{triplets}}\,\sin\theta\,\Delta\theta}$$

The $\sin\theta$ **cone correction** removes the geometric bias toward $90°$: the number of directions on a sphere between $\theta$ and $\theta+d\theta$ is $dN = 2\pi\sin\theta\,d\theta$, so an isotropic distribution would otherwise appear weighted toward $90°$.

**Key features**
- Peaks identify preferred angles: $104.5°$ (water), $109.5°$ (tetrahedral), $120°$ (planar)
- Broad peaks indicate flexibility; narrow peaks indicate rigid geometry
- Bimodal distributions signal coexisting conformations

In [ ]:
# Angle A1-A2-A3 with vertex at A2
adf = funcs.ADF(
    traj,
    at_g1=[o_idx[0]],    # first atom
    at_g2=[h_idx[0]],    # vertex
    at_g3=[o_idx[1]],    # third atom
    nbins=200,
    range=(0, 180),
)
adf_results = adf.calculate(cone_correction=True, plot=True)


In [ ]:
# Combined time-series + ADF panel
adf_results = adf.calculate(cone_correction=True, com_plot=True)

theta_peak = adf_results.x[adf_results.adf.argmax()]
print(f'Most probable angle: {theta_peak:.1f} deg')


---
## 4. Dihedral Distribution Function (DDF)

### Theory

The dihedral (torsion) angle $\phi$ for a four-atom chain $A_1$-$A_2$-$A_3$-$A_4$ is the angle between the $A_1$-$A_2$-$A_3$ plane and the $A_2$-$A_3$-$A_4$ plane:

$$\phi = \operatorname{atan2}\!\bigl(|\mathbf{b}_2|\,\mathbf{b}_1\cdot(\mathbf{b}_2\times\mathbf{b}_3),\ (\mathbf{b}_1\times\mathbf{b}_2)\cdot(\mathbf{b}_2\times\mathbf{b}_3)\bigr), \quad \phi\in(-180^\circ,180^\circ]$$

where $\mathbf{b}_i = \mathbf{r}_{i+1} - \mathbf{r}_i$ are the bond vectors.

**Key features**
- $\phi \approx \pm 60^\circ$: gauche; $\phi \approx 180^\circ$: trans
- Relative peak areas give rotamer population fractions
- Peak width is inversely related to the torsional barrier height
- A flat distribution indicates essentially free rotation

In [ ]:
# at_groups: list of [A1, A2, A3, A4] index lists, one per dihedral to track
dihedral_groups = [
    [c_idx[0], c_idx[1], c_idx[2], c_idx[3]],   # C-C-C-C backbone dihedral
]

ddf = funcs.DDF(traj, at_groups=dihedral_groups, nbins=180, range=(-180, 180))
ddf_results = ddf.calculate(plot=True)


In [ ]:
# Combined time-series + DDF panel
ddf_results = ddf.calculate(com_plot=True)


---
## 5. Combined Distribution Function (CDF)

### Theory

The CDF is the 2-D joint probability density of two structural observables $x_1$ and $x_2$ computed for the same atom group:

$$\mathrm{CDF}(x_1, x_2) = P(x_1, x_2) \propto \sum_t \delta(x_1 - x_1^{(t)})\,\delta(x_2 - x_2^{(t)})$$

The canonical application is the $r_{DA}$-$\angle DHA$ CDF for hydrogen bonds, which reveals the joint geometry basin and separates true H-bonds (short $r_{DA}$, large $\angle DHA$) from weakly interacting contacts.

**Key features**
- Elongated ridges indicate correlated variables
- Multiple density basins indicate coexisting structural motifs
- Marginal distributions recover the original 1-D RDF / ADF

In [ ]:
# Compute RDF and ADF for the same atom triplet
rdf_da = funcs.RDF(traj, at_g1=[o_idx[0]], at_g2=[o_idx[1]], nbins=150, range=(0.0, 6.0))
adf_dha = funcs.ADF(
    traj,
    at_g1=[o_idx[0]], at_g2=[h_idx[0]], at_g3=[o_idx[1]],
    nbins=150, range=(0, 180),
)
rdf_res = rdf_da.calculate()
adf_res = adf_dha.calculate(cone_correction=True)

# Construct the 2-D CDF
cdf = funcs.CDF(rdf_res, adf_res, names=[r'$r_{OO}$ (A)', r'$\angle OHO$ (deg)'])
cdf_results = cdf.calculate()


---
## 6. Scalar Distribution Function (SDF)

### Theory

The SDF computes the normalised probability density $g(\delta)$ for any pre-computed scalar observable $\delta$ extracted from the trajectory:

$$g(\delta) = \frac{\mathrm{count}(\delta,\delta+\Delta\delta)}{N\,\Delta\delta}$$

where $N$ is the total number of observations, so that $\int g(\delta)\,d\delta = 1$.

Unlike the RDF or ADF (which compute the observable internally), the SDF accepts a **pre-computed array** -- making it a general-purpose probability density estimator for any per-frame quantity: bond lengths, coordination numbers, order parameters, potential energies, etc.

**Key features**
- Peak position: most probable value of $\delta$
- Weighted mean: $\langle\delta\rangle = \int \delta\,g(\delta)\,d\delta$
- FWHM: thermal fluctuation magnitude around the preferred value
- `com_plot=True`: time-series + distribution on a shared $\delta$-axis (useful for checking equilibration)

In [ ]:
# Pre-compute a scalar observable: O-O distance between two oxygens
oo_dists = np.array([frame.dist(o_idx[0], o_idx[1], mic=True) for frame in traj.frames])

sdf = funcs.SDF(
    traj,
    scaler=oo_dists,
    nbins=150,
    range=(2.0, 6.0),
)
sdf_results = sdf.calculate(plot=True, x_label=r'$r_{OO}$ (A)')


In [ ]:
# Combined time-series + SDF panel
sdf_results = sdf.calculate(
    com_plot=True,
    x_label=r'$r_{OO}$ (A)',
    y_label=r'$g(r_{OO})$',
)

peak = sdf_results.x[sdf_results.sdf.argmax()]
mean = np.average(sdf_results.x, weights=sdf_results.sdf)
print(f'Peak position : {peak:.4g}')
print(f'Weighted mean : {mean:.4g}')


---
## 7. Vector Reorientation Dynamics (VRD)

### Theory

The VRD module computes the Legendre-polynomial autocorrelation of a normalised bond vector $\hat{\mathbf{u}}(t)$:

$$C_l(t) = \langle P_l[\hat{\mathbf{u}}(0)\cdot\hat{\mathbf{u}}(t)]\rangle$$

The Legendre polynomials are

$$P_1(x)=x,\qquad P_2(x)=\tfrac{1}{2}(3x^2-1),\qquad P_3(x)=\tfrac{1}{2}(5x^3-3x)$$

$C_1(t)$ is related to infrared line shapes and NMR $T_1$ relaxation.  $C_2(t)$ is measured by depolarised Raman and dielectric spectroscopy.  Both decay from 1 to 0 as the vector loses orientational memory.

The decay is fitted by the **Kohlrausch-Williams-Watts (KWW) stretched exponential**:

$$C_l(t) = \exp\!\left[-\left(\frac{t}{\tau}\right)^\beta\right]$$

$\beta\in(0,1]$ quantifies dynamical heterogeneity: $\beta=1$ is single-exponential; $\beta<1$ indicates a distribution of relaxation times.  The effective reorientation time:

$$\tau_{\mathrm{eff}} = \frac{\tau}{\beta}\,\Gamma\!\left(\frac{1}{\beta}\right)$$

**Key features**
- $C_2$ decays more slowly than $C_1$ ($\tau_2 \approx 3\tau_1$ for isotropic diffusion)
- Plot $\ln C_l$ vs $t$ to distinguish exponential from stretched-exponential behaviour
- Comparing different $l$ reveals the rotational diffusion mechanism

In [ ]:
# spec = [at_g1, at_g2] -- bond vector u = mean(pos(at_g2)) - pos(at_g1)
vrd = funcs.VRD(
    traj=traj,
    spec=[o_idx[:1], h_idx[:2]],   # O-H bond vector (first water molecule)
    sampling=50,                    # use every 50th frame as t=0 origin
    l=2,                            # Legendre order (1, 2, or 3)
    nframes=500,                    # correlation window length (frames)
)
vrd_results = vrd.calculate()


In [ ]:
# KWW parameters from the fit
print(f'tau      = {vrd_results.tau:.4f} ps')
print(f'beta     = {vrd_results.beta:.4f}')
print(f'tau_eff  = {vrd_results.tau_eff:.4f} ps')

# Log-scale plot to inspect the stretched-exponential shape
t_axis = np.arange(len(vrd_results.C_t)) * traj.timestep / 1000   # ps
fig, ax = plt.subplots()
ax.semilogy(t_axis, vrd_results.C_t, label=r'$C_2(t)$')
ax.semilogy(t_axis, vrd_results.C_t_fit, '--', label='KWW fit')
ax.set_xlabel(r'$t$ (ps)')
ax.set_ylabel(r'$C_2(t)$')
ax.legend()
plt.tight_layout()
plt.show()


---
## 8. Van Hove Correlation Function (VHCF)

### Theory

The Van Hove function decomposes atomic motion into self (single-particle) and distinct (pair) parts:

$$G(r,\tau) = G_s(r,\tau) + G_d(r,\tau)$$

#### Self part ($i=j$)

$$G_s(r,\tau) = \frac{1}{N}\sum_{i=1}^N\bigl\langle\delta\bigl(r - |\mathbf{r}_i(t+\tau) - \mathbf{r}_i(t)|\bigr)\bigr\rangle_t$$

Short $\tau$: narrow Gaussian near $r=0$ (caged motion).  Long $\tau$: broadening Gaussian (Fickian diffusion).  Deviations from Gaussianity are quantified by the **non-Gaussian parameter**:

$$\alpha_2(\tau) = \frac{3\langle r^4(\tau)\rangle}{5\langle r^2(\tau)\rangle^2} - 1$$

$\alpha_2=0$ for purely Gaussian dynamics; a peak at crossover time $\tau^*$ signals dynamical heterogeneity and transient trapping.

#### Distinct part ($i\ne j$)

$$G_d(r,\tau) = \frac{1}{N}\sum_{i\ne j}\bigl\langle\delta\bigl(r - |\mathbf{r}_j(t+\tau) - \mathbf{r}_i(t)|\bigr)\bigr\rangle_t$$

At $\tau=0$: $G_d(r,0) = \rho\,g(r)$ (recovers the RDF).  As $\tau\to\infty$: $G_d\to\rho$ (loss of structural memory).

> **Important**: the self-part requires **unwrapped** (continuous) coordinates.  Do **not** call `wrap2box()` before using `VHCF` in self mode.

In [ ]:
# Self-part: load trajectory WITHOUT wrap2box()
traj_raw = trj.Trajectory(timestep=5, data='trajectory.xyz', index=':', cell=cell)
traj_raw.calib()   # remove CoM drift only

vhcf_self = funcs.VHCF(
    traj_raw,
    at_g1=o_idx,        # atoms to track
    r_range=[0.0, 6.0],
    tau_sep=10,          # spacing between time origins (frames)
    tau_max=100,         # maximum tau (frames)
    nbins=100,
)
self_results = vhcf_self.calculate(plot=True)
vhcf_self.summary()


In [ ]:
# Non-Gaussian parameter alpha_2(tau)
r2 = self_results.mean_r2   # <r^2(tau)>
r4 = self_results.mean_r4   # <r^4(tau)>
alpha2 = 3*r4 / (5*r2**2) - 1
tau_ps = self_results.tau_axis

fig, ax = plt.subplots()
ax.plot(tau_ps, alpha2)
ax.axhline(0, color='k', lw=0.8, ls='--')
ax.set_xlabel(r'$\tau$ (ps)')
ax.set_ylabel(r'$\alpha_2(\tau)$')
ax.set_title('Non-Gaussian parameter')
plt.tight_layout()
plt.show()


In [ ]:
# Distinct-part VHCF (supply at_g2; wrapped traj is fine)
vhcf_dist = funcs.VHCF(
    traj,
    at_g1=o_idx,
    at_g2=h_idx,     # at_g2 != at_g1 activates the distinct mode
    r_range=[0.0, 8.0],
    tau_sep=10,
    tau_max=100,
    nbins=100,
)
dist_results = vhcf_dist.calculate(plot=True)


---
## 9. Dimer Analysis

### Theory

A *dimer* is any pair of atoms (or molecular CoMs) satisfying geometry criteria in a given frame.  Three modes are supported:

| Mode | `at_g1` format | `criteria` | Use case |
|------|---------------|------------|----------|
| Atom-atom distance | `[i, j, ...]` flat ints | `[r_max, None]` | contact pairs |
| CoM-CoM distance | `[[i,j,...], ...]` lists | `[r_max, None]`, `com=True` | molecular pairs |
| H-bond (D-H-A) | `[[D, [H1,H2,...]], ...]` | `[r_DA_max, theta_min]` | H-bonds |

Distance criteria: float (Angstrom), `'1.02vdw_sum'`, or `'1.1coval_sum'`.  Angle criteria: float (degrees) or flexible string `'120f'` that scales with instantaneous D-H bond length.

#### Dimer Lifetime-Displacement Distribution Function (DLDDF)

$$\mathrm{DLDDF}(\tau,\Delta r) = P(\text{lifetime}=\tau,\ |\Delta\mathbf{r}_{CoM}|=\Delta r)$$

**Interpretation**: short $\tau$ + small $\Delta r$ = thermal fluctuations; long $\tau$ + small $\Delta r$ = strongly bound / caged pairs; diagonal ridge = Brownian escape.

### `dimers`: detect dimer pairs in each frame

In [ ]:
# Build donor list: [[D_idx, [H1_idx, H2_idx]], ...]
# Assumes water molecules: O at o_idx[i], H at h_idx[2i] and h_idx[2i+1]
n_waters = min(4, len(o_idx))
water_donors = [[o_idx[i], [h_idx[2*i], h_idx[2*i+1]]] for i in range(n_waters)]
water_acceptors = o_idx[:8]

hb = dimer.dimers(
    traj,
    at_g1=water_donors,
    at_g2=water_acceptors,
    criteria=['1.02vdw_sum', '120f'],
)
hb.make_pairs()
hb.pair_filter(mic=True, filename='hbonds.json')

print('Frame 0:', hb.results.dimers[0])


In [ ]:
# Number of H-bonds per frame
key = 'pairs' if 'pairs' in hb.results.dimers[0] else 'd_h_a_pairs'
nhb = []
for d in hb.results.dimers:
    p = d[key]
    if p == []:
        nhb.append(0)
    elif all(isinstance(x, int) for x in p):
        nhb.append(1)
    else:
        nhb.append(len(p))

t_hb = np.array([d['frame'] for d in hb.results.dimers]) * traj.timestep / 1000
fig, ax = plt.subplots()
ax.plot(t_hb, nhb, lw=0.8)
ax.set_xlabel(r'$t$ (ps)')
ax.set_ylabel('Number of H-bonds')
ax.xaxis.set_minor_locator(MultipleLocator(10))
plt.tight_layout()
plt.show()


### `DLDDF`: lifetime-displacement 2-D histogram

In [ ]:
dimer_settings = {
    'at_g1': water_donors,
    'at_g2': water_acceptors,
    'criteria': ['1.02vdw_sum', '120f'],
}

dlddf = dimer.DLDDF(
    traj,
    dimer_settings=dimer_settings,
    nbins=100,
    range=np.array([(0.0, 2.0), (0.0, 5.0)]),   # [(tau_min,tau_max ps), (Dr_min,Dr_max A)]
)
dlddf.dimer_res(filename='hbonds.json')   # reuses JSON written above

dlddf_results = dlddf.calculate(
    nframes=200,
    mic=True,
    plot=True,
    xlims=[0, 2.0],
    ylims=[0, 5.0],
    levels=25,
)


---
## 10. H-bond Lifetime from Autocorrelation

### Theory

The **continuous H-bond survival autocorrelation** $C(t)$ gives the probability that a bond present at $t=0$ is still intact at time $t$ (no intermittent breaking allowed):

$$C(t) = \frac{\langle h(0)\,h(t)\rangle}{\langle h(0)^2\rangle}, \qquad C(0)=1, \quad C(t)\to 0$$

$C(t)$ is averaged over non-overlapping windows of $n_{\mathrm{frames}}$ frames.  The input is the JSON file produced by `dimers.pair_filter(filename=...)`.

The decay is fitted with a **biexponential model**:

$$C(t) = A\,e^{-t/\tau_1} + (1-A)\,e^{-t/\tau_2}$$

- $\tau_1$ (fast): librational / geminate-reformation dynamics
- $\tau_2$ (slow): structural H-bond breaking

The **effective lifetime** is the area under $C(t)$:

$$\tau_{\mathrm{eff}} = \int_0^\infty C(t)\,dt = A\tau_1 + (1-A)\tau_2$$

In [ ]:
# Compute C(t) from the JSON written by pair_filter
t_ct, C_mean, C_err = dimer.auto_cor(
    data='hbonds.json',
    nframes=1000,              # frames per averaging window
    skip=100,                  # stride within each window
    index=':',                 # use all frames
    timestep=traj.timestep,    # fs
)

fig, ax = plt.subplots()
ax.errorbar(t_ct, C_mean, yerr=C_err, fmt='o', ms=3, capsize=3)
ax.set_xlabel(r'$t$ (ps)')
ax.set_ylabel(r'$C(t)$')
ax.set_title('H-bond survival autocorrelation')
plt.tight_layout()
plt.show()


In [ ]:
# Biexponential fit
t_fit, C_fit, params = dimer.bi_exp_fit(
    t_ct, C_mean,
    amp1=0.5, tau1=0.05, tau2=0.5,   # initial guesses (ps)
)
A, tau1, tau2 = params

fig, ax = plt.subplots()
ax.errorbar(t_ct, C_mean, yerr=C_err, fmt='o', ms=3, capsize=3, label='data')
ax.plot(t_fit, C_fit, label='biexponential fit')
ax.set_xlabel(r'$t$ (ps)')
ax.set_ylabel(r'$C(t)$')
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Effective H-bond lifetime: tau_eff = integral_0^inf C(t) dt = A*tau1 + (1-A)*tau2
tau_eff = dimer.calc_tau(A, tau1, tau2)

print(f'Fitted parameters:')
print(f'  A       = {A:.4f}')
print(f'  tau1    = {tau1:.4f} ps  (fast/librational)')
print(f'  tau2    = {tau2:.4f} ps  (slow/structural)')
print(f'  tau_eff = {tau_eff:.4f} ps')
